In [1]:
import json
import re
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
fs = sorted(Path("../data/json2").glob("*.json"))
print(f"{len(fs)} threads")

10257 threads


In [3]:
from IPython.display import display
# load output from
df3 = pd.read_parquet("../outputs/df_links4.parquet")
df3

,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url
title,,,,,,,,
Worth the Candle,1453,110,110,[{'author_flair_text': 'Self-Appointed Court S...,[https://reddit.com/r/rational/comments/f1rj3l...,2017-07-28 13:17:36,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...
"Alexander Wales - The Metropolitan Man, Shadows of the Limelight",1408,22,22,[{'author_flair_text': 'Time flies like an arr...,[https://reddit.com/r/rational/comments/8mqrd0...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales]
A Practical Guide to Evil,1034,94,94,[{'author_flair_text': 'Ankh-Morpork City Watc...,[https://reddit.com/r/rational/comments/dutkzh...,2016-04-05 18:53:21,2024-08-09 10:10:21,"[https://practicalguidetoevil.wordpress.com/, ..."
Worm,925,83,83,[{'author_flair_text': 'Emergency Mustelid Hol...,[https://reddit.com/r/rational/comments/f1rj3l...,2014-01-27 02:09:42,2024-11-22 21:39:11,"[https://parahumans.wordpress.com/, https://pa..."
Mother of Learning,880,93,93,"[{'author_flair_text': 'Utopian Smut Peddler',...",[https://reddit.com/r/rational/comments/dutkzh...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...
...,...,...,...,...,...,...,...,...
Heartbreaker,-6,1,1,[{'author_flair_text': 'https://i.imgur.com/OQ...,[https://reddit.com/r/rational/comments/48akta...,2016-02-29 20:57:08,2016-02-29 20:57:08,[https://parahumans.wordpress.com/2012/01/17/b...
picking RPG clothes based on maxing stats instead of whether they match or not,-6,2,2,"[{'author_flair_text': None, 'body': 'It seems...",[https://reddit.com/r/rational/comments/9h1454...,2018-09-19 02:26:27,2018-09-19 04:51:09,[https://youtu.be/xq1tN9jZI80]
Orthogonality thesis,-8,2,2,"[{'author_flair_text': None, 'body': 'I don't ...",[https://reddit.com/r/rational/comments/3vc0si...,2015-12-04 18:17:07,2016-07-11 16:27:30,[https://wiki.lesswrong.com/wiki/Orthogonality...


## Extra get a llm summary of each link [WIP]

Grab all md's that mention a story, ask claude to summarize

We could also get total karma per mention

In [4]:
import dotenv
from anycache import anycache

dotenv.load_dotenv()
from openai import OpenAI

client = OpenAI()

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
cost = 0.150 / 1e6

In [5]:
# load all md posts 
md_posts = []
fs = sorted(Path("../data/cache2").glob("*.md"))
for f in fs:
    s = f.open().read()
    md_posts.append(s)


# order by date
def md2date(s: str) -> str:
    return s.split('* Created: ')[1].split('\n')[0]

md_posts = sorted(md_posts, key=md2date)
md2date(md_posts[0]), md2date(md_posts[-1])

('2009-11-25T02:34:03', '2024-12-23T15:00:14')

In [6]:

def get_post_context(urls: List[str], char_budget=100000, min_size=1000) -> str:
    # TODO maybe I should just get comment with links, and children?
    assert len(urls) > 0
    # from comments
    # return df3.loc[title].comments

    # or I could just all markdowns with

    # TODO use langchain chunking?

    matches = []
    for ii, post in enumerate(md_posts):
        for url in urls:
            if url in post:
                matches.append(post)
                break

    budget_pp = char_budget / len(matches)
    budget_pp = max(budget_pp, min_size)
    s = ""
    for i in range(len(matches)):
        post = matches[i]

        for url in urls:
            if url in post:
                ind = post.index(url)

        i0 = int(max(0, ind - budget_pp // 4))
        i1 = int(min(len(post), ind + budget_pp // 4 * 3))
        post_chunk = post[i0:i1]
        if i0 > 0:
            post_chunk = "..." + post_chunk
        if i1 < len(post):
            post_chunk = post_chunk + "..."

        s += f"\n\n----- Thread {ii} -----\n\n" + post_chunk

    # if too large get first N//2 and last N//2
    if len(s) > char_budget:
        s = s[:char_budget // 2] + "..." + s[-char_budget // 2:]
    return s


# url = df3.url[0].split('\n')
# print(url)
# c = get_context(url, 400000)
# print(c[:1000])

In [7]:
df3.iloc[-1].url

array(['https://sciencetrends.com/is-there-a-replicability-crisis-in-psychology-new-study-says-its-complicated/'],
      dtype=object)

In [8]:
# QC test with long and short context
# urls = ['https://archiveofourown.org/works/11478249/chapters/25740126',
#        'https://www.royalroad.com/fiction/25137/worth-the-candle',
#        'http://archiveofourown.org/works/11478249?view_full_work=true',
#        'http://archiveofourown.org/works/11478249/chapters/25740126',
#        'https://archiveofourown.org/works/11478249']
# c = get_post_context(urls, 400000)
# print(urls)
# print(c)

# # urls = ['https://www.amazon.com/Becoming-Batman-Possibility-Paul-Zehr/dp/0801890632']
# # c = get_post_context(urls, 400000)
# # print(urls)
# # print(c)


In [9]:


from pydantic import BaseModel, Field


class FictionInfo(BaseModel):
    title: str
    description: str = Field(description="A few paragraphs of very concise, informative, dense, description of the work")
    tags: List[str] = Field(
        description="""Long list of common descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)
        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)
        Content notes (grimdark, romance, harem, queer, funny, NSFW)
        """
    )

    # status: Optional[str] = Field(description="complete/ongoing/hiatus/abandoned")
    # type: str = Field(description='e.g. fanfiction, original, comic, etc.')

    reviews_quotes: List[str] = Field(
        description="Directly and fully quote excerpts from every single users' comments about the fiction"
    )
    reviews_summary: Optional[str] = Field(
        description="Structured, dry, and concise summary of reviews"
    )
    reccomendations: Optional[str] = Field(
        description="Why users recommend it"
    )
    disrecommendations: Optional[str] = Field(
        description="Why users disrecommend it"
    )
    why: Optional[str] = Field(
        description="Why/when might readers of r/rational like it"
    )
    if_you_liked_x_you_will_like_this: List[str] = Field(
        description="Fans of X will also like the work. Exhaustively list ALL of X mentioned in the discussions."
    )

    quality: float = Field(
        # ge=0.0, le=10.0,
        description="Overall user sentiment out of 10"
    )
    rationality: Optional[float] = Field(
        # ge=0.0, le=10.0,
        description="Systematic worldbuilding, character competence, logical consistency. Where HPMOR is a 10 and Worm is a 5."
    )
    rating_writing: Optional[float]
    rating_plot: Optional[float]
    rating_character: Optional[float]
    rating_worldbuilding: Optional[float]


f_cache_llm = Path("../outputs/.anycache3")


# @anycache(f_cache_llm)
def get_llm_summary(title: str, urls: str, context: str):
    chat_completion = client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": "You are Gwern Branwern, an internet librarian who specializes in rational fiction. You are summarising community reccomendations from r/rational into a dry, informative, concise, and structured format for your own personal notes. Because it's private you can be consise, frank, and opinionated.",
            },
            {
                "role": "user",
                "content": f"""Using the given structure, summarize the parts of these discussions where they talk about {title} (urls: {urls}).

### Context:

{context}""",
            },
        ],
        model="gpt-4o-mini",
        response_format=FictionInfo,
    )

    return chat_completion.choices[0].message.parsed.__dict__

In [10]:
# import shutil
# shutil.rmtree(f_cache_llm, ignore_errors=True)

In [11]:
from openai.lib._pydantic import to_strict_json_schema

to_strict_json_schema(FictionInfo)

{'properties': {'title': {'title': 'Title', 'type': 'string'},
  'description': {'description': 'A few paragraphs of very concise, informative, dense, description of the work',
   'title': 'Description',
   'type': 'string'},
  'tags': {'description': 'Long list of common descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)\n        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)\n        Content notes (grimdark, romance, harem, queer, funny, NSFW)\n        ',
   'items': {'type': 'string'},
   'title': 'Tags',
   'type': 'array'},
  'reviews_quotes': {'description': "Directly and fully quote excerpts from every single users' comments about the fiction",
   'items': {'type': 'string'},
   'title': 'Reviews Quotes',
   'type': 'array'},
  'reviews_summary': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'description': 'Structured, dry, and concise summary of reviews',
   'title': 'Reviews Summary'},
  'rec

In [14]:
llm_info = []


l = min(1000, len(df3))
for i in tqdm(range(0, l)):
    title = df3.index[i]
    urls = df3.iloc[i].url

    context = get_post_context(urls, char_budget=50000)
    tokens = len(enc.encode(context))
    print(
        f"Input Tokens: {tokens}. Input Cost: {cost * tokens:.4f} USD, for title={title} with urls={urls}"
    )

    llm_data = get_llm_summary(title, urls, context)

    llm_data["title2"] = title
    llm_data["url"] = urls

    if i==0:
        print(f"Content: {context[:1000]}")
        display(llm_data)

    llm_info.append(llm_data)

  0%|          | 0/1000 [00:00<?, ?it/s]

Input Tokens: 13920. Input Cost: 0.0021 USD, for title=Worth the Candle with urls=['https://archiveofourown.org/works/11478249/chapters/25740126'
 'https://www.royalroad.com/fiction/25137/worth-the-candle'
 'http://archiveofourown.org/works/11478249?view_full_work=true'
 'http://archiveofourown.org/works/11478249/chapters/25740126'
 'https://archiveofourown.org/works/11478249'
 'http://daystareld.com/podcast/rationally-writing-42/'
 'https://www.royalroad.com/fiction/25137/worth-the-candle/chapter/366577/taking-the-fall']
Content: 

----- Thread 10262 -----

## [RT][WIP] Worth the Candle, Ch 1

* Author: u/cthulhuraejepsen  *Fruit flies like a banana**
* URL: http://archiveofourown.org/works/11478249/chapters/25740126
* Score: 32

* Created: 2017-07-14T05:23:38

### Post:

[Link to content](http://archiveofourown.org/works/11478249/chapters/25740126)

### Comments:

> **u/cthulhuraejepsen** [+9]  *Fruit flies like a banana* (49 minutes later)
> 
> This is a self-insert litRPG portal fa

{'title': 'Worth the Candle',
 'description': '*Worth the Candle* is a self-insert litRPG portal fantasy written by u/cthulhuraejepsen. The narrative follows Juniper Smith, a teenage boy unexpectedly transported to a fantasy world inspired by the multiple tabletop RPGs he has run. The story incorporates game mechanics into its plot, offering a unique character sheet and interactive gameplay elements that impact character development and world interaction. As Juniper navigates this new reality, he grapples with existential themes, including the nature of agency and personal identity amidst the fantastical chaos.',
 'tags': ['web serial',
  'fantasy',
  'scifi',
  'litRPG',
  'self-insert',
  'portal fantasy',
  'light',
  'humor',
  'grimdark',
  'harem'],
 'reviews_quotes': ['"This is quickly becoming my favorite litrpg. You need to work a little harder to catch up to the wandering inn."',
  '"I\'m reasonably confident that her beauty is abnormal..."',
  '"Hell of a name for a chapter 

Input Tokens: 12769. Input Cost: 0.0019 USD, for title=Alexander Wales - The Metropolitan Man, Shadows of the Limelight with urls=['https://www.patreon.com/alexanderwales']
Input Tokens: 13919. Input Cost: 0.0021 USD, for title=A Practical Guide to Evil with urls=['https://practicalguidetoevil.wordpress.com/'
 'https://practicalguidetoevil.wordpress.com/table-of-contents/'
 'https://practicalguidetoevil.wordpress.com/summary/'
 'https://practicalguidetoevil.wordpress.com/2015/03/25/prologue/'
 'https://practicalguidetoevil.wordpress.com'
 'https://practicalguidetoevil.wordpress.com/2017/08/30/interlude-commanders/'
 'https://practicalguidetoevil.wordpress.com/2018/07/02/court-iii/'
 'https://practicalguidetoevil.wordpress.com/2018/06/01/court-ii/'
 'https://practicalguidetoevil.wordpress.com/2018/05/02/court-i/'
 'https://practicalguidetoevil.wordpress.com/2016/03/02/chapter-14-situation/'
 'https://practicalguidetoevil.wordpress.com/2018/03/07/epilogue-3/'
 'https://practicalguidetoev

In [13]:
df_llm = pd.DataFrame(llm_info)
df_llm
# also join with df4

# then display as table

,title,description,tags,reviews_quotes,reviews_summary,reccomendations,disrecommendations,why,if_you_liked_x_you_will_like_this,quality,rationality,rating_writing,rating_plot,rating_character,rating_worldbuilding,title2,url
0,Worth the Candle,Worth the Candle is a self-insert litrpg porta...,"[web serial, litrpg, fantasy, self-insert, pro...","[""If you haven't read this, it's a self-insert...",Readers praise the narrative for its unique bl...,The community highly recommends this work for ...,Some users caution that readers not typically ...,Fans of rational fiction and those who enjoy t...,"[Worm, Mother of Learning, The Wandering Inn, ...",9.0,8.0,8.0,9.0,8.0,9.0,Worth the Candle,[https://archiveofourown.org/works/11478249/ch...
1,"The Metropolitan Man, Shadows of the Limelight","""The Metropolitan Man"" and ""Shadows of the Lim...","[web serial, fantasy, rational, superhero, com...","[""Feels a lot like a Sanderson book"", ""Effecti...",The sentiment towards Alexander Wales' works i...,Readers appreciate the complexity of the magic...,Some fans note the infrequency of updates and ...,"Fans of intricate magic systems, moral complex...","[Mistborn, Worm, The Mother of Learning, Worth...",9.0,8.0,8.0,9.0,9.0,8.0,"Alexander Wales - The Metropolitan Man, Shadow...",[https://www.patreon.com/alexanderwales]
2,A Practical Guide to Evil,A Practical Guide to Evil is a web serial that...,"[web serial, fantasy, rational, metanarrative,...","[""Probably one of the better written ones out ...",Readers appreciate the depth of character deve...,"The story's intricate plotting, compelling cha...",Some readers find the prose stylistically awkw...,Fans of rational fiction who enjoy complex cha...,"[Worm, HPMOR, Mother of Learning, Unsong, The ...",9.0,8.0,7.0,9.0,9.0,8.0,A Practical Guide to Evil,"[https://practicalguidetoevil.wordpress.com/, ..."
3,Worm,Worm is a web serial written by John C. 'Wildb...,"[web serial, scifi, superhero, dark, gritty, r...",[I wouldn't really describe Worm as a rational...,"Readers find Worm a compelling, albeit dark de...",Readers recommend Worm for its in-depth charac...,Some users caution that it can be emotionally ...,Fans of rationality-themed works may appreciat...,"[Harry Potter and the Methods of Rationality, ...",9.0,6.0,8.0,9.0,9.0,7.0,Worm,"[https://parahumans.wordpress.com/, https://pa..."
